# Notebook 04: Analysis & Paper Tables

Dravidian Extension — Telugu / Tamil / Kannada, generic vs. localized conditions.

Produces all statistics, tables, and figures for the paper.

**Inputs:**
- `../results/checkpoints/raw_responses.json` (from `01_run_inference.py`) — used for script-fidelity, since `judge_scores.json` only contains records that already passed the fidelity filter
- `../data/judge_scores.json` (from `02_judge_responses.py`)
- `../data/embed_distances.json` (from `03_embed_distance.py`)

**Outputs:** `../results/final_table.csv`, figures in `../results/figures/`

**Note:** this pipeline has no `religion` field and no Hindi/Punjabi — those belong to the *reference* repo this project was adapted from. Here the two axes are `lang` (`en`/`te`/`ta`/`kn`) and `version` (`generic`/`localized`).

In [ ]:
!pip install -q scipy pandas matplotlib seaborn

In [ ]:
import json, os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE     = Path(".")
DATA_DIR = BASE / "../data"
CKPT_DIR = BASE / "../results/checkpoints"
RES_DIR  = BASE / "../results"
FIG_DIR  = RES_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

raw_path   = CKPT_DIR / "raw_responses.json"
judge_path = DATA_DIR / "judge_scores.json"
embed_path = DATA_DIR / "embed_distances.json"

for p in (raw_path, judge_path, embed_path):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. Run 01_run_inference.py, 02_judge_responses.py, "
            f"and 03_embed_distance.py first."
        )

with open(raw_path)   as f: raw_data   = json.load(f)
with open(judge_path) as f: judge_data = json.load(f)
with open(embed_path) as f: embed_data = json.load(f)

rdf = pd.DataFrame(raw_data)      # model, scenario_id, dimension, version, lang, region, response, script_ok, truncated
jdf = pd.DataFrame(judge_data)    # model, scenario_id, dimension, version, lang, script_ok, truncated, stance, reasoning, response_snippet
edf = pd.DataFrame(embed_data)    # model, scenario_id, dimension, version, lang_pair, anchor, cosine_sim, drift

LANGS           = ["en", "te", "ta", "kn"]
DRAVIDIAN_LANGS = ["te", "ta", "kn"]
VERSIONS        = ["generic", "localized"]

# Valid judge scores only (stance == 0 means the judge saw an empty/off-topic/
# refused response; stance == -1 means the judge call itself failed).
jdf = jdf[jdf["stance"] > 0].copy()

print(f"Raw records:        {len(rdf)}")
print(f"Judge records (valid stance): {len(jdf)}")
print(f"Embed records:      {len(edf)}")
print(f"Models: {sorted(rdf['model'].unique())}")
print(f"Dimensions: {sorted(rdf['dimension'].unique())}")

In [ ]:
# ── TABLE 1: Script Fidelity ───────────────────────────────────────────────────
# How many models can actually write Telugu / Tamil / Kannada script?
# NOTE: computed from raw_responses.json, not judge_scores.json — 02_judge_
# responses.py already restricts itself to script_ok==True records before
# judging, so judge_scores.json alone would show 100% fidelity by construction.

fidelity_rows = []
for model in rdf["model"].unique():
    for lang in DRAVIDIAN_LANGS:
        sub = rdf[(rdf["model"] == model) & (rdf["lang"] == lang)]
        pass_rate = sub["script_ok"].mean() * 100 if len(sub) else float("nan")
        fidelity_rows.append({"Model": model, "Language": lang, "Script Fidelity %": round(pass_rate, 1)})

fidelity_df = pd.DataFrame(fidelity_rows).pivot(index="Model", columns="Language", values="Script Fidelity %")
fidelity_df = fidelity_df[[c for c in DRAVIDIAN_LANGS if c in fidelity_df.columns]]
print("\nTABLE 1: Script Fidelity")
print(fidelity_df.to_string())

In [ ]:
# ── TABLE 2: Cross-lingual Value Drift (judge scores, generic condition) ──────
# For each model × dimension: mean stance in EN vs TE vs TA vs KN.
# Restricted to version=="generic": that's the only condition with a single,
# unambiguous EN anchor per (model, scenario_id) — the localized condition has
# three region-specific EN anchors that collide on (model, scenario_id, lang=en)
# once script_ok/truncated filtering is applied, so it can't be paired the same
# way here. Localized-vs-generic is handled separately in the H3 section below,
# where pairing is on (model, scenario_id, lang) across version and doesn't
# need the EN anchor at all.

jdf_generic = jdf[jdf["version"] == "generic"]

drift_rows = []
models = jdf_generic["model"].unique()
dims   = jdf_generic["dimension"].unique()

for model in models:
    for dim in dims:
        sub = jdf_generic[(jdf_generic["model"] == model) & (jdf_generic["dimension"] == dim)]
        mean_en = sub[sub["lang"] == "en"]["stance"].mean()
        mean_te = sub[sub["lang"] == "te"]["stance"].mean()
        mean_ta = sub[sub["lang"] == "ta"]["stance"].mean()
        mean_kn = sub[sub["lang"] == "kn"]["stance"].mean()
        drift_rows.append({
            "model":       model,
            "dimension":   dim,
            "stance_EN":   round(mean_en, 2),
            "stance_TE":   round(mean_te, 2),
            "stance_TA":   round(mean_ta, 2),
            "stance_KN":   round(mean_kn, 2),
            "drift_EN_TE": round(abs(mean_en - mean_te), 2),
            "drift_EN_TA": round(abs(mean_en - mean_ta), 2),
            "drift_EN_KN": round(abs(mean_en - mean_kn), 2),
        })

drift_df = pd.DataFrame(drift_rows)
print("\nTABLE 2: Cross-lingual Value Drift (judge scores, generic condition)")
print(drift_df.to_string(index=False))

In [ ]:

# ── STATISTICAL TESTS (H1, H2) ────────────────────────────────────
# Exclude models with script fidelity failure from H1/H2, determined
# empirically from Table 1 (below a threshold in ANY Dravidian language),
# rather than hardcoding model names from a different project's model list.

FIDELITY_THRESHOLD = 50.0  # percent
SCRIPT_FAIL_MODELS = set(
    fidelity_df[(fidelity_df < FIDELITY_THRESHOLD).any(axis=1)].index
)
SCRIPT_CAPABLE = [m for m in jdf_generic["model"].unique() if m not in SCRIPT_FAIL_MODELS]
jdf_stats = jdf_generic[jdf_generic["model"].isin(SCRIPT_CAPABLE)]
print(f"Models excluded from H1/H2 (script failure, <{FIDELITY_THRESHOLD}% in some language): {SCRIPT_FAIL_MODELS or 'none'}")
print(f"Models included in H1/H2 ({len(SCRIPT_CAPABLE)}): {list(SCRIPT_CAPABLE)}")

print("\n── H1: Is drift(EN→Dravidian) significantly > 0 across script-capable models? ──")

def paired_stances(target_lang):
    en_vals, tgt_vals = [], []
    for _, row in jdf_stats[jdf_stats["lang"] == "en"].iterrows():
        tgt_row = jdf_stats[
            (jdf_stats["model"] == row["model"]) &
            (jdf_stats["scenario_id"] == row["scenario_id"]) &
            (jdf_stats["lang"] == target_lang)
        ]
        if len(tgt_row):
            en_vals.append(row["stance"])
            tgt_vals.append(tgt_row.iloc[0]["stance"])
    return np.array(en_vals), np.array(tgt_vals)

def cohens_d(a, b):
    return (np.mean(a) - np.mean(b)) / np.sqrt((np.std(a)**2 + np.std(b)**2) / 2)

drift_by_lang = {}
for lang in DRAVIDIAN_LANGS:
    en_arr, tgt_arr = paired_stances(lang)
    if len(en_arr) < 2:
        print(f"H1 EN vs {lang.upper()}: not enough paired records, skipping")
        continue
    stat, p = stats.wilcoxon(en_arr, tgt_arr)
    drift = np.abs(en_arr - tgt_arr)
    d = cohens_d(tgt_arr, en_arr)
    drift_by_lang[lang] = drift
    print(f"H1 EN vs {lang.upper()}: W={stat:.1f}, p={p:.4f}, mean_drift={drift.mean():.2f}, Cohen's d={d:.3f}")

print("\n── H2: Does drift(EN→Dravidian) differ by language? ──")
lang_pairs = [(a, b) for i, a in enumerate(DRAVIDIAN_LANGS) for b in DRAVIDIAN_LANGS[i+1:]]
for a, b in lang_pairs:
    if a in drift_by_lang and b in drift_by_lang and len(drift_by_lang[a]) == len(drift_by_lang[b]):
        stat_h2, p_h2 = stats.wilcoxon(drift_by_lang[a], drift_by_lang[b])
        d_h2 = cohens_d(drift_by_lang[a], drift_by_lang[b])
        print(f"H2 {a.upper()} vs {b.upper()}: W={stat_h2:.1f}, p={p_h2:.4f}, Cohen's d={d_h2:.3f}, "
              f"mean drift {a.upper()}={drift_by_lang[a].mean():.3f} vs {b.upper()}={drift_by_lang[b].mean():.3f}")
    else:
        print(f"H2 {a.upper()} vs {b.upper()}: unpaired sample sizes differ, comparing distributions with Mann-Whitney instead")
        if a in drift_by_lang and b in drift_by_lang:
            stat_h2, p_h2 = stats.mannwhitneyu(drift_by_lang[a], drift_by_lang[b], alternative="two-sided")
            print(f"  U={stat_h2:.1f}, p={p_h2:.4f}")

In [ ]:

# ── H3: Does localizing the scenario (generic → localized) change drift? ──────
# This is this repo's own H3 (see the built-in summaries at the end of
# 02_judge_responses.py / 03_embed_distance.py) — NOT an "Aya vs other models"
# comparison, which belonged to a different project's model roster.
#
# Judge-score version: paired on (model, scenario_id, lang) across version.
# This key is safe here — unlike the EN anchor rows, native-language (te/ta/kn)
# rows never collide across region.

print("── H3 (judge stance): generic vs localized, per model × language ──")
h3_judge_rows = []
for model in jdf["model"].unique():
    for lang in DRAVIDIAN_LANGS:
        gen = jdf[(jdf["model"] == model) & (jdf["lang"] == lang) & (jdf["version"] == "generic")]
        loc = jdf[(jdf["model"] == model) & (jdf["lang"] == lang) & (jdf["version"] == "localized")]
        merged = pd.merge(gen[["scenario_id", "stance"]], loc[["scenario_id", "stance"]],
                           on="scenario_id", suffixes=("_generic", "_localized"))
        if len(merged) >= 2:
            stat, p = stats.wilcoxon(merged["stance_generic"], merged["stance_localized"])
        else:
            stat, p = np.nan, np.nan
        h3_judge_rows.append({
            "model": model, "lang": lang, "n_paired": len(merged),
            "mean_generic": round(merged["stance_generic"].mean(), 2) if len(merged) else np.nan,
            "mean_localized": round(merged["stance_localized"].mean(), 2) if len(merged) else np.nan,
            "wilcoxon_p": round(p, 4) if p == p else np.nan,
        })
h3_judge_df = pd.DataFrame(h3_judge_rows)
print(h3_judge_df.to_string(index=False))

# Embedding-drift version: edf already carries a correctly-anchored "drift"
# column per version (03_embed_distance.py resolves the region-specific EN
# anchor internally), so we can pair directly on (model, scenario_id, lang_pair).
print("\n── H3 (embedding drift): generic vs localized, per model × language pair ──")
h3_embed_rows = []
for model in edf["model"].unique():
    for lp in [f"en→{l}" for l in DRAVIDIAN_LANGS]:
        gen = edf[(edf["model"] == model) & (edf["lang_pair"] == lp) & (edf["version"] == "generic")]
        loc = edf[(edf["model"] == model) & (edf["lang_pair"] == lp) & (edf["version"] == "localized")]
        merged = pd.merge(gen[["scenario_id", "drift"]], loc[["scenario_id", "drift"]],
                           on="scenario_id", suffixes=("_generic", "_localized"))
        if len(merged) >= 2:
            stat, p = stats.wilcoxon(merged["drift_generic"], merged["drift_localized"])
        else:
            stat, p = np.nan, np.nan
        h3_embed_rows.append({
            "model": model, "lang_pair": lp, "n_paired": len(merged),
            "mean_drift_generic": round(merged["drift_generic"].mean(), 3) if len(merged) else np.nan,
            "mean_drift_localized": round(merged["drift_localized"].mean(), 3) if len(merged) else np.nan,
            "wilcoxon_p": round(p, 4) if p == p else np.nan,
        })
h3_embed_df = pd.DataFrame(h3_embed_rows)
print(h3_embed_df.to_string(index=False))

In [ ]:
# ── FIGURE 1: Drift heatmap by model × dimension (generic condition) ─────────

for lang_code, col in [("te", "drift_EN_TE"), ("ta", "drift_EN_TA"), ("kn", "drift_EN_KN")]:
    pivot = drift_df.pivot_table(values=col, index="model", columns="dimension", aggfunc="mean")
    n_models = len(pivot)
    fig, ax = plt.subplots(figsize=(9, max(4, n_models * 0.65)))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd",
                linewidths=0.5, ax=ax, vmin=0, vmax=2)
    ax.set_title(f"Cross-lingual Value Drift EN→{lang_code.upper()} by Model & Dimension (generic)", fontsize=12)
    ax.set_xlabel("Hofstede Dimension")
    ax.set_ylabel("Model")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"fig1_drift_heatmap_{lang_code}.pdf", bbox_inches="tight")
    plt.show()
    print(f"Figure 1 ({lang_code}) saved.")

In [ ]:
# ── FIGURE 2: EN vs TE vs TA vs KN stance per dimension (boxplot, generic) ────

dims = jdf_generic["dimension"].unique()
fig, axes = plt.subplots(1, len(dims), figsize=(3.5 * len(dims), 4), sharey=True)
if len(dims) == 1:
    axes = [axes]

for ax, dim in zip(axes, dims):
    sub = jdf_generic[jdf_generic["dimension"] == dim]
    data = [sub[sub["lang"] == l]["stance"].values for l in LANGS]
    ax.boxplot(data, labels=[l.upper() for l in LANGS])
    ax.set_title(dim, fontsize=9)
    ax.set_ylim(0.5, 5.5)
    ax.set_ylabel("Stance (1=Western, 5=South Asian)" if ax is axes[0] else "")

plt.suptitle("Value Stance Distribution by Language and Dimension (generic condition)", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_stance_boxplot.pdf", bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

In [ ]:
# ── FIGURE 3: Embedding drift EN→TE / EN→TA / EN→KN by model ──────────────
# Pooled across generic + localized, matching 03_embed_distance.py's own summary.

embed_model_lang = edf.groupby(["model", "lang_pair"])["drift"].mean().reset_index()
target_pairs = [f"en→{l}" for l in DRAVIDIAN_LANGS]
embed_pivot = embed_model_lang[embed_model_lang["lang_pair"].isin(target_pairs)].pivot(
    index="model", columns="lang_pair", values="drift"
)
embed_pivot = embed_pivot[[c for c in target_pairs if c in embed_pivot.columns]]

fig, ax = plt.subplots(figsize=(8, 4))
embed_pivot.plot(kind="bar", ax=ax, color=["steelblue", "darkorange", "seagreen"], edgecolor="black")
ax.set_title("Semantic Embedding Drift by Model (LaBSE)", fontsize=12)
ax.set_ylabel("Mean Cosine Drift (1 - similarity)")
ax.set_xlabel("Model")
ax.legend(title="Language Pair")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_embed_drift.pdf", bbox_inches="tight")
plt.show()
print("Figure 3 saved.")

In [ ]:
# ── Save final CSV for paper ───────────────────────────────────────

RES_DIR.mkdir(parents=True, exist_ok=True)
drift_df.to_csv(RES_DIR / "final_table.csv", index=False)

summary = drift_df.groupby("model")[["drift_EN_TE", "drift_EN_TA", "drift_EN_KN"]].mean().round(2)
print("\n── MAIN RESULT: Mean drift by model (generic condition) ──")
print(summary.to_string())
print(f"\nAll results saved to {RES_DIR}/")